# 08_03 Reading both ways: does a BiLSTM beat an LSTM on headlines?

The book trained an LSTM and a BiLSTM on its fake-news titles and got 91.5 and 90.1 percent. This notebook
asks the same question of news headlines, on all 47,500 training headlines, and then asks a sharper one: how
much of what the LSTM learned depends on word order at all?

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-08-remembering-across-a-sentence", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'torch': 'torch',
           'sklearn': 'scikit-learn'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import os
import time
import torch
import headlines
from nlpcheck import ask, guess, reveal, check_08_03

torch.set_num_threads(4)
data = headlines.load_split()
X_train, y_train, X_test, y_test = data["X_train"], data["y_train"], data["X_test"], data["y_test"]
print(len(X_train), "training headlines,", len(X_test), "held out")

## 1. Recall

**r5.** Why did the RNN in the last notebook fail when it read the padding? (a) padding ids are unknown words,
(b) its final state came after many steps of zeros, which the error signal had to cross to reach any real word,
(c) padding makes the batch too large

**r6.** What does `pack_padded_sequence` need, besides the padded batch? (a) each row's real length, (b) the
vocabulary, (c) the labels

In [ ]:
ask("r5", "")
ask("r6", "")

## 2. The LSTM and the BiLSTM on every headline

The same packed LSTM as before, now on all the training headlines for 3 epochs, and a BiLSTM, the same
`SequenceClassifier` with `bidirectional=True`, trained the same way. Training both takes several minutes, so
the session trained them in the background while you read the chapter, with exactly this recipe (seed 0, 3
epochs); `pretrain.trained` loads each in a second, or, if the background job has not finished, trains it now
and says so.

First, predict: will the BiLSTM beat the LSTM by more than one point of accuracy?

In [ ]:
guess("bilstm_wins", None)   # "yes" or "no" 

In [ ]:
import pretrain
results = {}
lstm = pretrain.trained("lstm", X_train, y_train)
bilstm = pretrain.trained("bilstm", X_train, y_train)
for name, model in (("lstm", lstm), ("bilstm", bilstm)):
    results[name] = headlines.evaluate(model, X_test, y_test)
    print(f"{name:7} {results[name]}")
reveal("bilstm_wins", "yes" if results["bilstm"]["accuracy"] > results["lstm"]["accuracy"] + 0.01 else "no")

About 0.906 for the LSTM and 0.902 for the BiLSTM: no gain, at twice the cost. The book found the same on its
fake-news titles, 91.5 against 90.1. A classifier that reads to the end already has every word in its final
state, so the backward direction adds a second reading rather than new information. BiLSTMs earn their cost when
a decision is made at every word, as in tagging.

## 3. The honest baseline

Lab 07's lesson was to measure a model that ignores order before crediting order: the average of the word
embeddings, trained the same way, also in the background (about 20 seconds if it trains now).

In [ ]:
bag = pretrain.trained("bag", X_train, y_train)
results["bag"] = headlines.evaluate(bag, X_test, y_test)
print("bag    ", results["bag"])

About 0.914: the bag beats both recurrent networks, as it did in Lab 07, and trains in a fraction of the time.
On this task, which words appear decides almost everything. That leaves a sharper question than "does order
help": how much is the LSTM using order at all?

## 4. How much does the LSTM use word order?

Shuffle the words inside every test headline, keeping the padding at the end, and score both trained models
again, without retraining. Your part is the one line that shuffles: `torch.randperm(k, generator=g)` is the
numbers 0 to k - 1 in a random order, and indexing `row[:k]` with it reorders the real ids. Predict first how
many points of accuracy the LSTM loses.

In [ ]:
guess("shuffle_drop", None)   # points of accuracy, for example 20

In [ ]:
g = torch.Generator().manual_seed(0)
X_shuffled = X_test.clone()
for i, row in enumerate(X_shuffled):
    k = int((row != 0).sum())
    X_shuffled[i, :k] = row[:k]   # YOUR CODE HERE: put these k real ids in a random order, using generator=g
print("a headline:          ", X_test[0][:10].tolist())
print("the same, shuffled:  ", X_shuffled[0][:10].tolist())
results["lstm_shuffled"] = headlines.evaluate(lstm, X_shuffled, y_test)
results["bag_shuffled"] = headlines.evaluate(bag, X_shuffled, y_test)
drop = 100 * (results["lstm"]["accuracy"] - results["lstm_shuffled"]["accuracy"])
print(f"LSTM: {results['lstm']['accuracy']:.3f} in order, {results['lstm_shuffled']['accuracy']:.3f} shuffled")
print(f"bag:  {results['bag']['accuracy']:.3f} in order, {results['bag_shuffled']['accuracy']:.3f} shuffled")
reveal("shuffle_drop", round(drop))

In [ ]:
os.makedirs("out", exist_ok=True)
json.dump(results, open("out/08_03_results.json", "w"), indent=1)
check_08_03()

The LSTM drops from about 0.906 to 0.893, a little over one point, and the bag does not move, because an average
does not know the order it was given. So the LSTM did learn something from word order, and it needed very little
of it: a shuffled headline is still mostly the right words. Order carries the meaning in sentences such as "not
good" or "rates rise as inflation falls", and those are where a reading-in-order model is worth its cost. It is
also where, from Chapter 10, attention takes over.

## 5. Exit ticket

Explain it back, in the cell below, in two sentences of your own: when would you expect a BiLSTM to beat an
LSTM, and why not here?

**x4.** A BiLSTM is (a) an LSTM with two layers, (b) two LSTMs over the same words, one reading forwards and one
backwards, (c) an LSTM trained twice

**x5.** Why did the bag's accuracy not change when the words were shuffled? (a) an average of embeddings is the
same for any order of the same words, (b) the shuffle missed the bag's test set, (c) the bag was retrained

In [ ]:
my_explanation = ""
ask("x4", "")
ask("x5", "")